In [1]:
import pandas as pd
df = pd.read_csv('train.csv')

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_selector
from sklearn.impute import SimpleImputer
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

# feature engineer
def engineered_features(df):
    """針對重要特徵進交互作用與缺失模的特徵工程。
    
    參考依據 (importance gain)：
    1. 變數貢獻高度集中：
    - stress_level 與 physical_activity 貢獻了絕大部分的預測增益。

    2. 缺失值的資訊量：
    - 在預處理實驗過程中，觀察到原始資料帶有NaN狀態的特徵（如 stress_level_nan 351.9、
      physical_activity_nan 247.2）重要性顯著超越 BMI、心率等傳統生理指標。
    - 證實此資料集的缺失並非隨機發生 (MNAR)，「未填寫/資料遺漏」本身即帶有強烈的風險行為特徵。

    工程策略：
    1. 類別交互特徵：
    - 建立 stress x activity 交叉組合，手動建構非線性情境（例如高壓力+低運動 vs 高壓力+高運動）。
      因根據邏輯來看運動可產生腦內啡，可是十運動時，大腦會釋放腦內啡。它能結合大腦中的阿片受體，
      抑制疼痛訊號，同時帶來愉悅與放鬆感，抵銷高壓帶來的焦慮情緒。
    
    2. 缺失模式(目前因表現不佳，暫時刪除缺失型態)
    - 針對三大核心指標 (sleep, stress, activity)的缺失值。衍生出「缺失數量」與「缺失型態」，
      表示受測者的問卷填寫行為模式。
    """
    df = df.copy()

    # 1. 缺失數量
    missing_sleep = df["sleep_duration"].isna().astype(int)
    missing_stress = df['stress_level'].isna().astype(int)
    missing_activity = df['physical_activity_level'].isna().astype(int)
    df['missing_key_counts'] = missing_sleep + missing_stress +missing_activity

    # 2. 壓力與活動量的交互作用
    has_both = df['stress_level'].notna() & df['physical_activity_level'].notna()
    df['stress_activity_interaction'] = np.where(
      has_both,
      df['stress_level'].astype(str)
      + '_'
      + df['physical_activity_level'].astype(str),
      np.nan, 
    )
    # # 3. 缺失型態
    # df['missing_pattern'] = (
    #     missing_sleep.astype(str)
    #     + '_'
    #     + missing_stress.astype(str)
    #     + '_'
    #     + missing_activity.astype(str)
    # )
    return df

class  GroupSleepDiffTransformer(BaseEstimator, TransformerMixin):
    '''計算各種壓力與活動，睡眠時間相對群體的平均偏離度
    單看睡眠小時數』的絕對值無法完全反映個體的生理風險。例如：同樣睡眠 6 小時，在『高壓+久坐
    的極限耗損情境下可能屬於嚴重不足；但在低壓+規律運動的情境下卻可能是相對充裕的。透過計算情境相對偏離度，能精準捕捉該個體在特定
    生理負擔下是否屬於相對較為脆弱的人。

    **更新:可擴充其他變數分組的偏離度，並修正如果sleep_duration是nan，那麼差距就要是nan
    '''

    def __init__(
        self,
        group_col='stress_activity_interaction',
        target_cols=None,
    ):
        self.group_col = group_col
        self.target_cols = (
            target_cols
            if target_cols
            else ['sleep_duration']
        )

    def fit(self, X, y=None):
        if hasattr(X, 'columns'):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)

            self.group_means_ = {}
            self.global_means_ = {}
        for col in self.target_cols:
            self.group_means_[col] = X.groupby(self.group_col)[col].mean().to_dict()
            self.global_means_[col] = X[col].mean()
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.target_cols:
            group_mean = (
                X[self.group_col]
                .map(self.group_means_[col])
                .fillna(self.global_means_[col])
            )
            X[f'{col}_diff_from_group_mean'] = X[col] - group_mean
        return X

    def get_feature_names_out(self, input_features=None):
        input_features = (
            input_features if input_features is not None else self.feature_names_in_
        )
        new_cols = [f'{col}_diff_from_group_mean' for col in self.target_cols]
        return np.asarray(list(input_features) + new_cols, dtype=object)

# ordinal
ordinal_cols = ['stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol']
stress_order = ['missing', 'low', 'medium', 'high']
sleep_order = ['missing', 'poor', 'average', 'good']
activity_order = ['missing', 'sedentary', 'moderate', 'active']
smoke_order = ['missing', 'no', 'occasional', 'yes']

ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value = 'missing')),
    ('encoder', OrdinalEncoder(
        categories=[stress_order, sleep_order, activity_order, smoke_order],
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))    
])

# nominal 
nominal_cols = ['diet_type', 'gender', 'stress_activity_interaction']
nominal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value = 'missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])


# 欄位轉換

column_trans =  ColumnTransformer(
    transformers = [
        ('num', 'passthrough', make_column_selector(dtype_include=['number'])),
        ('ordinal_col', ordinal_transformer, ordinal_cols),
        ('nominal_col', nominal_transformer, nominal_cols),
        
])


# 整合資料處理管線
preprocessor = Pipeline(steps=[
    ('feature_engineer', FunctionTransformer(engineered_features)),
    ('sleep_diff', GroupSleepDiffTransformer(
        target_cols=['sleep_duration', 'bmi', 'step_count', 'exercise_duration']
    )),
    ('column_transformer', column_trans)
])

preprocessor.set_output(transform='pandas')


,steps,"[('feature_engineer', ...), ('sleep_diff', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,func,<function eng...001CB6F70C5E0>
,inverse_func,None
,validate,False
,accept_sparse,False
,check_inverse,True
,feature_names_out,None
,kw_args,None


In [3]:
# define the data
from sklearn.preprocessing import LabelEncoder
# X
X = df.loc[:, [c for c in df.columns if c not in ['id', 'health_condition']]]
# y
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['health_condition'])
# split the data
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [13]:
test = preprocessor.fit_transform(X_train)
test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 552070 entries, 313415 to 507278
Data columns (total 34 columns):
 #   Column                                                     Non-Null Count   Dtype  
---  ------                                                     --------------   -----  
 0   num__sleep_duration                                        491368 non-null  float64
 1   num__heart_rate                                            545874 non-null  float64
 2   num__bmi                                                   540984 non-null  float64
 3   num__calorie_expenditure                                   509948 non-null  float64
 4   num__step_count                                            540949 non-null  float64
 5   num__exercise_duration                                     546559 non-null  float64
 6   num__water_intake                                          517205 non-null  float64
 7   num__missing_key_counts                                    552070 non-null  int64  

In [ ]:
import json
from lightgbm import LGBMClassifier
from sklearn.utils.class_weight import compute_sample_weight
with open('best_lgb_params2.json', 'r')as f:
    lgb_params = json.load(f)

mod1 = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('lgb', LGBMClassifier(**lgb_params))
])
mod1.fit(X_train, y_train, lgb__sample_weight = np.sqrt(compute_sample_weight('balanced', y_train)))

NameError: name 'X_train' is not defined

In [28]:
from sklearn.metrics import classification_report
mod1_train_predict = mod1.predict(X_train)
mod1_test_predict = mod1.predict(X_test)
report_mod1_train = classification_report(y_train, mod1_train_predict, digits=4)
report_mod1_test = classification_report(y_test, mod1_test_predict, digits=4)
print(f'train: {report_mod1_train}\n test: {report_mod1_test}')

train:               precision    recall  f1-score   support

           0     0.9922    0.9682    0.9800    474049
           1     0.8259    0.9507    0.8839     31842
           2     0.8320    0.9519    0.8879     46179

    accuracy                         0.9658    552070
   macro avg     0.8834    0.9569    0.9173    552070
weighted avg     0.9692    0.9658    0.9668    552070

 test:               precision    recall  f1-score   support

           0     0.9876    0.9625    0.9749    118512
           1     0.8066    0.9249    0.8617      7961
           2     0.7951    0.9226    0.8541     11545

    accuracy                         0.9570    138018
   macro avg     0.8631    0.9366    0.8969    138018
weighted avg     0.9611    0.9570    0.9583    138018



In [18]:
# gain
booster = mod1.named_steps['lgb'].booster_
imp_df = pd.DataFrame({
    'feature': booster.feature_name(),
    'importance': booster.feature_importance(importance_type='gain'),
}).sort_values('importance', ascending=False)

imp_df['clean_feature'] = imp_df['feature'].str.split('__').str[-1]
display(imp_df[['clean_feature', 'importance']])

,clean_feature,importance
0,sleep_duration,1.588140e+06
12,stress_level,1.291959e+06
27,stress_activity_interaction_low_active,1.011206e+06
14,physical_activity_level,3.356818e+05
8,sleep_duration_diff_from_group_mean,3.258244e+05
7,missing_key_counts,8.454015e+04
2,bmi,7.906162e+04
33,stress_activity_interaction_missing,7.392841e+04
9,bmi_diff_from_group_mean,4.897245e+04
4,step_count,4.795933e+04


In [21]:
# optuna
# 超參數調整
import numpy as np
import optuna
from lightgbm import LGBMClassifier
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight


optuna.logging.set_verbosity(optuna.logging.INFO)
def objective(trial):
  # 1. 定義超參數搜尋空間 (Search Space)
  lgb_params = {
      'objective': 'multiclass',
      'num_class': 3,
      'metric': 'multi_logloss',
      'boosting_type': 'gbdt',
      # 學習率與樹的數量
      'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.08),
      'n_estimators': trial.suggest_int('n_estimators', 200, 800, step=50),
      # 樹結構防守核心
      'num_leaves': trial.suggest_int('num_leaves', 20, 63),
      'max_depth': trial.suggest_int('max_depth', 4, 8),
      'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
      # 採樣與防過擬合
      'subsample': trial.suggest_float('subsample', 0.6, 0.9),
      'subsample_freq': 1,
      'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
      'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
      'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
      'random_state': 42,
      'n_jobs': -1,
  }

  skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
  cv_logloss = []

  # 2. 執行 5-Fold 交叉驗證
  for train_idx, val_idx in skf.split(X_train, y_train):
    X_tr, y_tr = X_train.iloc[train_idx], y_train[train_idx]
    X_va, y_va = X_train.iloc[val_idx], y_train[val_idx]    

    # 建立該 Fold 的 Pipeline
    pipeline = Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('lgb', LGBMClassifier(**lgb_params)),
        ]
    )

    tr_weights = np.sqrt(compute_sample_weight('balanced', y_tr))
    pipeline.fit(X_tr, y_tr, lgb__sample_weight=tr_weights)
    val_probs = pipeline.predict_proba(X_va)
    cv_logloss.append(log_loss(y_va, val_probs))

  return np.mean(cv_logloss)


# 3. 建立並啟動 Study
print('開始搜尋 LightGBM 最佳超參數...')
study = optuna.create_study(direction='minimize')
study.optimize(
    objective, n_trials=20)  
print('\n✅ 搜尋完成！')
print('最佳 LogLoss:', study.best_value)
print('最佳超參數組合:')
for k, v in study.best_params.items():
  print(f'  {k}: {v}')

[I 2026-07-29 22:19:59,880] A new study created in memory with name: no-name-ec2075e5-9206-4339-8e8d-f794eecb548f


開始搜尋 LightGBM 最佳超參數...


[I 2026-07-29 22:23:08,682] Trial 0 finished with value: 0.10624437158869342 and parameters: {'learning_rate': 0.060948433622075096, 'n_estimators': 400, 'num_leaves': 42, 'max_depth': 7, 'min_child_samples': 60, 'subsample': 0.6576251655956544, 'colsample_bytree': 0.784323818714223, 'reg_alpha': 0.0031087140784000124, 'reg_lambda': 0.004671217732574681}. Best is trial 0 with value: 0.10624437158869342.
[I 2026-07-29 22:25:54,362] Trial 1 finished with value: 0.11043830301858551 and parameters: {'learning_rate': 0.021150166983421745, 'n_estimators': 400, 'num_leaves': 48, 'max_depth': 6, 'min_child_samples': 83, 'subsample': 0.7143948069432006, 'colsample_bytree': 0.6788337791654184, 'reg_alpha': 1.8043239707387908e-08, 'reg_lambda': 0.001800576093425933}. Best is trial 0 with value: 0.10624437158869342.
[I 2026-07-29 22:28:12,555] Trial 2 finished with value: 0.10843938438900622 and parameters: {'learning_rate': 0.05546470513312379, 'n_estimators': 400, 'num_leaves': 27, 'max_depth': 


✅ 搜尋完成！
最佳 LogLoss: 0.1028448697515318
最佳超參數組合:
  learning_rate: 0.07929189493794446
  n_estimators: 800
  num_leaves: 35
  max_depth: 8
  min_child_samples: 100
  subsample: 0.607531439452638
  colsample_bytree: 0.652223471293691
  reg_alpha: 9.869153232464834e-07
  reg_lambda: 2.125577538130531e-08


In [22]:
import json
best_params = study.best_params
with open('best_lgb_any6_t20.json', 'w')  as f:
    json.dump(best_params, f, indent=4)

In [25]:
with open('best_lgb_any6_t20.json', 'r') as f:
    best_params = json.load(f)
print(best_params)

{'learning_rate': 0.07929189493794446, 'n_estimators': 800, 'num_leaves': 35, 'max_depth': 8, 'min_child_samples': 100, 'subsample': 0.607531439452638, 'colsample_bytree': 0.652223471293691, 'reg_alpha': 9.869153232464834e-07, 'reg_lambda': 2.125577538130531e-08}


In [26]:
import joblib
mod2 =  Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('lgb', LGBMClassifier(**best_params))
])

mod2.fit(X_train, y_train, lgb__sample_weight = np.sqrt(compute_sample_weight('balanced', y_train)))

,steps,"[('preprocessor', ...), ('lgb', ...)]"
,transform_input,None
,memory,None
,verbose,False
,steps,"[('feature_engineer', ...), ('sleep_diff', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,func,<function eng...0021F6C7EAE60>
,inverse_func,None
,validate,False


In [29]:
from sklearn.metrics import classification_report
mod2_train_predict = mod2.predict(X_train)
mod2_test_predict = mod2.predict(X_test)
report_mod2_train = classification_report(y_train, mod2_train_predict, digits=4)
report_mod2_test = classification_report(y_test, mod2_test_predict, digits=4)
print(f'train: {report_mod2_train}\n test: {report_mod2_test}')

train:               precision    recall  f1-score   support

           0     0.9940    0.9728    0.9833    474049
           1     0.8480    0.9644    0.9025     31842
           2     0.8563    0.9620    0.9060     46179

    accuracy                         0.9715    552070
   macro avg     0.8994    0.9664    0.9306    552070
weighted avg     0.9740    0.9715    0.9722    552070

 test:               precision    recall  f1-score   support

           0     0.9866    0.9652    0.9758    118512
           1     0.8146    0.9173    0.8629      7961
           2     0.8076    0.9172    0.8589     11545

    accuracy                         0.9584    138018
   macro avg     0.8696    0.9332    0.8992    138018
weighted avg     0.9617    0.9584    0.9595    138018



In [30]:
import joblib
final_mod =  Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('lgb', LGBMClassifier(**best_params))
])

final_mod.fit(X, y, lgb__sample_weight = np.sqrt(compute_sample_weight('balanced', y)))
joblib.dump(final_mod, 'health_lgb_diff_mod4_.pkl')

['health_lgb_diff_mod4_.pkl']